## Introduction

This notebook develops user interface (UI) menus options can be used to retrieve relevant WDL tasks for use in RAG.

We need user input in order to select the right WDL tasks to build a worfkflow. We need to know what kind of input data they have (e.g. paired-end DNA FASTQs), their analysis goals (e.g. variant calling), and if they prefer any specific tools (e.g. bwa-mem, strelka). Our ChromaDB RAG database 

We want to avoid free-text input for many reasons, inclding to prevent users from generating toxic output or leaking protected data. Therefore, we will give users preset options that can be mapped to metadata keywords in our database. In a point-and-click interface these would be dropdown menus. For this MVP we will use a command-line interface with numbered options.

## 1. Imports and loading database

In [4]:
import chromadb

def get_collection(chroma_dir, collection_name="wdl_tasks"):
    client = chromadb.PersistentClient(path=chroma_dir)
    return client.get_collection(name=collection_name)

collection = get_collection('../../data/chroma/')

## 2. Decide metadata to get user input for and filter on

What metadata-related information can the user provide? We should collect that and use it to keyword-filter WDL tasks.

Example of metadata for a single WDL task (some/all can be provided as context to the LLM):

```{python}
    metadatas=[{
        "tool": "strelka",
        "task": "strelka_germline",
        "topic": ["genomics", "dna_polymorphism"],
        "species": ["eukaryote"],
        "operation": "variant_calling",
        "input_sample_data_types": ["nucleic_acid_sequence_alignment", "data_index"],
        "input_sample_format_types": ["bam", "bai"],
        "output_sample_data_types": ["sequence_variations", "data_index"]
    }]
```

**I imagine we'll ask the user:**
- "What kind of sequencing data do you have?" (e.g. DNA, Bulk RNA, etc.)
- "What format is your data in?" (e.g. BAM, FASTQ, etc.)
- "What species is your data from?" (e.g. human, non-human eukaryote, etc.)
- "What kind of processing do you want done on this data?" (e.g. QC, Alignment, etc.) - NOTE: variant calling may require alignment first etc
- "Do you have any bioinformatics tool preferences?" (e.g. fastqc, bwa)

### Metadata categories relevant to the user interface:
- `tool`: User may have bioinformatics preferences (terms can be used as-is)
- `topic`: Whether the data is DNA/RNA/protein, etc.
- `species`: What species the data is from
- `operation`: What the user wants the WDL to do
- `input_sample_format_types`: Format usually tells us enough about the data type (e.g. FASTQ, BAM)

In [5]:
# Get all the metadata sections and look at first one as an example
all_metadata = collection.get(include=['metadatas'])['metadatas']
all_metadata[0]

{'tool': 'cnvkit',
 'operation': ['indexing'],
 'task': 'create_reference',
 'species': ['human', 'eukaryote'],
 'input_sample_format_types': ['bam', 'bai'],
 'output_sample_data_types': ['none'],
 'input_sample_data_types': ['nucleic_acid_sequence_alignment', 'data_index'],
 'topic': ['genomics', 'copy_number_variation']}

## 3. Get terms to be used as-is

Terms from categories:
- `tool`
- `input_sample_format_types`

First, a function to collect unique terms:

In [6]:
def get_unique_terms(category, all_meta):
    """Get all the unique terms used for a given metadata category 
    across all_meta (which should be entire database metadata)
    """
    unique_terms = set()
    for metadata in all_meta:
        terms_list = metadata.get(category, [])
        if isinstance(terms_list, list):
            unique_terms.update(terms_list)
        elif isinstance(terms_list, str):
            unique_terms.add(terms_list)
    return sorted(unique_terms)

**`tool`**

Tools we should omit from the 'bioinformatics tool preferences' menu (no choice for some operations, or not relevant to most users):
- aws-sso
- ena
- gatk (need to be task-level specific)
- gdc
- sjl
- sra


Some 'operation' options to consider:
- Download from SRA
- Download from ENA
- Convert file formats

Some 'input data type' and 'topic' related options to consider:
- Single-cell RNA
- Bulk RNA

In [7]:
tools_list = get_unique_terms("tool", all_metadata)
tools_list

['annotsv',
 'annovar',
 'aws-sso',
 'bcftools',
 'bedparse',
 'bedtools',
 'bowtie',
 'bowtie2',
 'bwa',
 'cellranger',
 'clair3',
 'cnvkit',
 'colabfold',
 'consensus',
 'deeptools',
 'deepvariant',
 'delly',
 'deseq2',
 'diamond',
 'ena',
 'esmfold',
 'fastp',
 'fastqc',
 'gatk',
 'gdc',
 'gffread',
 'glimpse2',
 'ichorcna',
 'jcast',
 'manta',
 'megahit',
 'multiqc',
 'rmats-turbo',
 'rseqc',
 'salmon',
 'samtools',
 'shapemapper',
 'sjl',
 'smoove',
 'sourmash',
 'spades',
 'sra',
 'star',
 'starling',
 'strelka',
 'trimgalore',
 'tritonnp',
 'varscan',
 'viennarna']

In [9]:
# GATK tasks a user may want to select
gatk_list = [
    'gatk: mutect2',
    'gatk: markduplicates',
    'gatk: baserecalibrator',
    'gatk: markduplicates',
    'gatk: haplotypecaller',
    'gatk: fastqtosam',
    'gatk: analyzesaturationmutagenesis'
    ]
tools_to_remove = {'aws-sso', 'ena', 'gatk', 'gdc', 'sjl', 'sra'}

updated_tool_list = [t for t in tools_list if t not in tools_to_remove] + gatk_list
updated_tool_list

['annotsv',
 'annovar',
 'bcftools',
 'bedparse',
 'bedtools',
 'bowtie',
 'bowtie2',
 'bwa',
 'cellranger',
 'clair3',
 'cnvkit',
 'colabfold',
 'consensus',
 'deeptools',
 'deepvariant',
 'delly',
 'deseq2',
 'diamond',
 'esmfold',
 'fastp',
 'fastqc',
 'gffread',
 'glimpse2',
 'ichorcna',
 'jcast',
 'manta',
 'megahit',
 'multiqc',
 'rmats-turbo',
 'rseqc',
 'salmon',
 'samtools',
 'shapemapper',
 'smoove',
 'sourmash',
 'spades',
 'star',
 'starling',
 'strelka',
 'trimgalore',
 'tritonnp',
 'varscan',
 'viennarna',
 'gatk: mutect2',
 'gatk: markduplicates',
 'gatk: baserecalibrator',
 'gatk: markduplicates',
 'gatk: haplotypecaller',
 'gatk: fastqtosam',
 'gatk: analyzesaturationmutagenesis']

**`input_sample_format_types`**

Formats we should omit because they're too specific or not meaningful (might be intermediate input for tasks):
- any
- bai
- binary_format
- configuration_file_format
- crai
- csi
- none
- sig
- tar_format
- tbi
- textual_format
- zip_format

In [20]:
formats_list = get_unique_terms("input_sample_format_types", all_metadata)
formats_list

['any',
 'bai',
 'bam',
 'bcf',
 'bed',
 'bigwig',
 'binary_format',
 'configuration_file_format',
 'crai',
 'cram',
 'csi',
 'csv',
 'directory',
 'fasta',
 'fastq',
 'gtf',
 'matrix',
 'none',
 'npz',
 'pileup',
 'sam',
 'sig',
 'tar_format',
 'tbi',
 'textual_format',
 'tsv',
 'vcf',
 'wig',
 'zip_format']

In [21]:
formats_to_remove = {
    'any', 'bai', 'binary_format', 'configuration_file_format', 'crai', 'csi',
    'none', 'sig', 'tar_format', 'tbi', 'textual_format', 'zip_format'}

updated_formats_list = [t for t in formats_list if t not in formats_to_remove]
updated_formats_list

['bam',
 'bcf',
 'bed',
 'bigwig',
 'cram',
 'csv',
 'directory',
 'fasta',
 'fastq',
 'gtf',
 'matrix',
 'npz',
 'pileup',
 'sam',
 'tsv',
 'vcf',
 'wig']

## 4. Map metadata terms to user-friendly ones where needed

Most metadata terms are from the EDAM ontology (to be consistent and descriptive). However, these are not terms researchers would use in daily life (e.g. "nucleic acid sequence alignment"). We need to come up with user-friendly terms for:

- `species` (not using EDAM)
- `topic`
- `operation`

**`species`**

In [16]:
species_list = get_unique_terms("species", all_metadata)
species_list

['eukaryote', 'human', 'prokaryote', 'virus']

In [17]:
# Let's be specific about the eukaryote
updated_species_list = ["non-human eukaryote" if x == "eukaryote" else x for x in species_list]
updated_species_list

['non-human eukaryote', 'human', 'prokaryote', 'virus']

**`topic`**

Map topic keywords to intput data being DNA, RNA, or protein. `topic` will also be used with `operation` below to filter tasks by analysis goal.

In [18]:
# Look at what we're working with
topic_list = get_unique_terms("topic", all_metadata)
topic_list

['any',
 'copy_number_variation',
 'data_quality_management',
 'dna_mutation',
 'dna_packaging',
 'dna_polymorphism',
 'epigenomics',
 'gene_expression',
 'genomics',
 'mapping',
 'metagenomics',
 'nucleic_acid_structure_analysis',
 'protein_disordered_structure',
 'protein_expression',
 'protein_structure_analysis',
 'proteomics',
 'public_health_and_epidemiology',
 'ribosome_profiling',
 'rna_splicing',
 'sequence_assembly',
 'sequence_features',
 'sequencing',
 'structural_variation',
 'transcriptomics']

Create a dictionary of mappings.

In [ ]:
# Group topics together with the kind of input data they apply to
topic_dict = {
    'dna': ['any', 'data_quality_management', 'sequencing', 'genomics', 
            'epigenomics', 'dna_packaging', 'dna_mutation', 
            'dna_polymorphism', 'metagenomics', 'copy_number_variation', 
            'nucleic_acid_structure_analysis', 'structural_variation', 
            'sequence_assembly', 'mapping', 'sequence_features'],
    'rna': ['any', 'data_quality_management', 'sequencing', 'transcriptomics', 
            'gene_expression', 'rna_splicing', 'ribosome_profiling', 'mapping',
            'sequence_features'],
    'protein': ['any', 'data_quality_management', 'sequencing', 'proteomics', 
                'protein_disordered_structure', 'protein_expression', 
                'protein_structure_analysis', 'mapping']
}

**`operation`**

User interface terms should allow user to convey their analysis goal(s). There will be overlap with topics also.

In [20]:
# Refresher on the topic terms:
topic_list

['any',
 'copy_number_variation',
 'data_quality_management',
 'dna_mutation',
 'dna_packaging',
 'dna_polymorphism',
 'epigenomics',
 'gene_expression',
 'genomics',
 'mapping',
 'metagenomics',
 'nucleic_acid_structure_analysis',
 'protein_disordered_structure',
 'protein_expression',
 'protein_structure_analysis',
 'proteomics',
 'public_health_and_epidemiology',
 'ribosome_profiling',
 'rna_splicing',
 'sequence_assembly',
 'sequence_features',
 'sequencing',
 'structural_variation',
 'transcriptomics']

In [21]:
# Look at the operation terms
operation_list = get_unique_terms("operation", all_metadata)
operation_list

['aggregation',
 'alternative_splicing_prediction',
 'annotation',
 'copy_number_variation_detection',
 'data_deposition',
 'data_filtering',
 'data_formatting',
 'data_handling',
 'data_retrieval',
 'file_handling',
 'indel_detection',
 'indexing',
 'mapping',
 'nucleic_acid_structure_analysis',
 'protein_structure_prediction',
 'quality_control',
 'quantification',
 'rna_secondary_structure_prediction',
 'rna_seq_quantification',
 'sequence_alignment',
 'sequence_assembly',
 'sequence_classification',
 'sequence_conversion',
 'sequence_trimming',
 'sequencing_quality_control',
 'splitting',
 'statistical_calculation',
 'variant_calling',
 'visualisation']

In [ ]:
# Map analysis goal phrases to operation and topic terms

analysis_goals = {
    'call variants (SNPs)': {'operation': [],
                             'topic': []},
    'call variants (structural)'
    'call variants (copy number)'
    'align to reference'
    'download data'
    'upload data'
    'perform quality control'
    'annotate variants'
    'annotate sequence features'
    'convert file types'
    'combine files'
    'measure gene expression'
    'predict protein structure'
    'predict alternative splicing'
    
}


['aggregation',
 'alternative_splicing_prediction',
 'annotation',
 'copy_number_variation_detection',
 'data_deposition',
 'data_filtering',
 'data_formatting',
 'data_handling',
 'data_retrieval',
 'file_handling',
 'indel_detection',
 'indexing',
 'mapping',
 'nucleic_acid_structure_analysis',
 'protein_structure_prediction',
 'quality_control',
 'quantification',
 'rna_secondary_structure_prediction',
 'rna_seq_quantification',
 'sequence_alignment',
 'sequence_assembly',
 'sequence_classification',
 'sequence_conversion',
 'sequence_trimming',
 'sequencing_quality_control',
 'splitting',
 'statistical_calculation',
 'variant_calling',
 'visualisation']

## 5. Write code to present menu options to user

In [ ]:
#TODO

## 6. Write logic that connects the inputs to each other

For example, based on format what operations do they need?

## 6. Connect inputs to the RAG database

Keyword filtering to retrieve tasks for the LLM. For now just return the tasks that would be retrieved (can hammer out the retrieval details later, such as choosing between ties etc.).

In [ ]:
# TODO